# 03 — Running the eval

Now that we have our evaluation dataset ready, it's time to build the core evaluation pipeline. This involves taking each test case, merging it with our prompt, feeding it to Claude, and then grading the results.

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

In [2]:
# Get the client and model from utils
from src.utils import get_client, get_model

client = get_client()
model = get_model()

In [3]:
# Helper functions
from src.utils import chat, add_user_message, add_assistant_message

## 1. The `run_prompt` function

Right now, we're keeping the prompt extremely simple. We're not including any formatting instructions, so Claude will likely return more verbose output than we need. We'll refine this later as we iterate on our prompt design.

In [ ]:
def run_prompt(test_case: dict[str, str]) -> str:
    """
    Merges the prompt and test case input, then returns the result
    
    Args:
        test_case (dict[str, str]): A dictionary containing the task and any
            additional information needed to solve the task.
    
    Returns:
        str: The output generated by the model after running the prompt.
    """
    
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages=messages, content=prompt)
    output = chat(
        messages=messages,
        client=client,
        model=model,
        temperature=0.0,
    )
    return output

## 2. The `run_test_case` Function

For now, we're using a hardcoded score of 10. The grading logic is where we'll spend significant time in upcoming sections, but this placeholder lets us test the overall pipeline.

In [5]:
def run_test_case(test_case: dict[str, str]) -> dict[str, str | int | dict]:
    """
    Calls run_prompt, then grades the result
    
    Args:
        test_case (dict[str, str]): A dictionary containing the task and any
            additional information needed to solve the task.
    
    Returns:
        dict[str, str | int | dict]: A dictionary containing the output, test case, and score.
    """
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

## 3. The `run_eval` Function

This function processes every test case in our dataset and collects all the results into a single list.

In [6]:
def run_eval(dataset: list[dict[str, str]]) -> list[dict[str, str | int | dict]]:
    """
    Loads the dataset and calls run_test_case with each case
    
    Args:
        dataset (list[dict[str, str]]): A list of dictionaries, each containing a task and any additional information needed to solve the task.
    
    Returns:
        list[dict[str, str | int | dict]]: A list of dictionaries, each containing the output, test case, and score.
    """
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

## 4. Running the Evaluation

The first time you run this, expect it to take some time - even with Claude Haiku, it can take around 30 seconds to process a full dataset. We'll cover optimization techniques later.

In [7]:
import json


with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\ndef extract_region_from_s3_url(s3_url):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        s3_url (str): S3 URL in the format 's3://bucket-name.region.amazonaws.com'\n    \n    Returns:\n        str: The AWS region name, or None if the URL format is invalid\n    \n    Examples:\n        >>> extract_region_from_s3_url('s3://my-bucket.us-west-2.amazonaws.com')\n        'us-west-2'\n        >>> extract_region_from_s3_url('s3://example-bucket.eu-central-1.amazonaws.com')\n        'eu-central-1'\n    \"\"\"\n    try:\n        # Remove the 's3://' prefix\n        if not s3_url.startswith('s3://'):\n            return None\n        \n        # Extract the hostname part\n        hostname = s3_url[5:]  # Remove 's3://'\n        \n        # Split by dots to get parts\n        parts = hostname.split('.')\n        \n        # Expected format: bucket